# 🐍 Python Learning Series: Notebook 04
## Python Logging Mastery: Handlers, Structured JSON, Log Rotation & Enterprise Observability

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rohit-Saini-Sfdc/learn-python/blob/main/04_logging_mastery.ipynb)

Welcome to Notebook 04 of the **Learn Python** series! 

In production software development, print statements (`print()`) are completely inadequate for tracking application health, debugging runtime errors, or auditing system transactions. Python's built-in `logging` module provides a flexible, event-driven framework for routing, filtering, formatting, and persisting system diagnostic messages.

---

### 🎯 What You Will Learn:
1. **The Flaws of `print()`**: Why production systems require severity levels, timestamps, contextual metadata, and non-blocking I/O.
2. **Log Hierarchy & Propagation**: How loggers form parent-child trees using dot notation (`app.services.auth`) and how log propagation works.
3. **Core Building Blocks**: Deep dive into `Logger`, `Handler`, `Formatter`, and `Filter`.
4. **Log Rotation Management**: Preventing disk exhaustion using `RotatingFileHandler` and `TimedRotatingFileHandler`.
5. **Structured JSON & Colored Logging**: Building custom formatters for local terminal debugging vs cloud-native observability (ELK, CloudWatch, Datadog).
6. **Enterprise `dictConfig` Architecture**: Configuring production applications declaratively using dictionary/YAML configurations.
7. **Lazy Formatting & Exception Tracking**: Performance optimization with deferred evaluation and capturing full stack trace context (`logger.exception()`).
8. **Real-World ETL Pipeline Project**: A complete end-to-end production data pipeline simulation with structured logging, file rotation, and error fallback.


## 📌 Module 0: Jupyter Notebook & Google Colab Logging Setup

In interactive environments like Jupyter Notebooks and Google Colab, the root logger is already pre-configured with default stream handlers (`ipykernel` / `google.colab`). Calling `logging.basicConfig()` directly without resetting handlers or using `force=True` (Python 3.8+) will cause new configurations to be ignored.

Let's write a utility function to reset loggers cleanly between our learning modules.


In [1]:
import logging
import sys
import os

def reset_logging():
    """Clear all handlers from the root logger and set default state for clean notebook execution."""
    root = logging.getLogger()
    for handler in root.handlers[:]:
        root.removeHandler(handler)
        handler.close()
    root.setLevel(logging.WARNING)

reset_logging()
print(f"Python Version: {sys.version.split()[0]}")
print("Logging environment successfully reset!")


Python Version: 3.14.7
Logging environment successfully reset!


## 📌 Module 1: Why `print()` is NOT Logging (The Core Problem)

Using `print()` statements in production code leads to major issues:
- **No Severity Levels**: All messages are treated equally (debug info vs critical failures).
- **No Destination Control**: Everything goes to standard output (`sys.stdout`) with no built-in file saving, log rotation, or remote streaming.
- **No Structured Metadata**: Lacks timestamps, line numbers, module names, thread IDs, or process context.
- **Performance Overhead**: `print()` eagerly formats strings and blocks execution.

### Logging Levels & Numeric Hierarchy:
| Level | Numeric Value | Use Case |
| :--- | :---: | :--- |
| **`CRITICAL`** | 50 | Serious error indicating the program itself may be unable to continue running. |
| **`ERROR`** | 40 | Due to a more serious problem, the software has not been able to perform some function. |
| **`WARNING`** | 30 | An indication that something unexpected happened (e.g., 'disk space low', fallback used). |
| **`INFO`** | 20 | Confirmation that things are working as expected (e.g., service started, task completed). |
| **`DEBUG`** | 10 | Detailed diagnostic information, typically of interest only when diagnosing problems. |
| **`NOTSET`** | 0 | When set on a logger, causes all events to be passed to its parent. |


In [2]:
reset_logging()

# Configure basic logging with level threshold set to INFO (20)
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(message)s',
    force=True
)

logger = logging.getLogger("demo_module_1")

# Test all 5 standard logging levels
logger.debug("This is a DEBUG message - ignored because threshold is INFO (20)")
logger.info("This is an INFO message - System operational.")
logger.warning("This is a WARNING message - High memory utilization detected (82%).")
logger.error("This is an ERROR message - Database connection timeout after 3 retries.")
logger.critical("This is a CRITICAL message - Service crash imminent! Memory exhausted.")


INFO - This is an INFO message - System operational.
WARNING - This is a WARNING message - High memory utilization detected (82%).
ERROR - This is an ERROR message - Database connection timeout after 3 retries.
CRITICAL - This is a CRITICAL message - Service crash imminent! Memory exhausted.


## 📌 Module 2: The Core Architecture & Logger Hierarchy

Python logging follows an object-oriented architecture built on 4 main pillars:
1. **Logger**: The entry point for application code to emit log records (`logger.info(...)`).
2. **Handler**: Directs log records to specific destinations (Console, File, Network Socket, Email).
3. **Formatter**: Specifies the layout and string presentation of log records.
4. **Filter**: Provides fine-grained control to filter records beyond simple log levels.

### Logger Hierarchy & Dot Notation
Loggers are arranged in a namespace hierarchy using dots (`.`). 
- Root Logger (`""`)
  - `app`
    - `app.database`
    - `app.api`
      - `app.api.auth`

Child loggers pass messages up to their parent's handlers unless `logger.propagate = False`.


In [3]:
reset_logging()

# Setup parent logger
parent_logger = logging.getLogger("app")
parent_logger.setLevel(logging.DEBUG)

# Add stream handler to parent logger only
handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter('[PARENT HANDLER] %(name)s | %(levelname)s | %(message)s')
handler.setFormatter(formatter)
parent_logger.addHandler(handler)

# Create child loggers using dot notation
auth_logger = logging.getLogger("app.api.auth")
db_logger = logging.getLogger("app.database")

print(f"Auth Logger Name: {auth_logger.name}")
print(f"Parent Logger Name: {auth_logger.parent.name}")
print(f"Effective Level of Auth Logger: {logging.getLevelName(auth_logger.getEffectiveLevel())}\n")

# Emit logs from child loggers - notice how they propagate to parent's handler!
auth_logger.info("User 'alice' authenticated successfully.")
db_logger.warning("Query execution took 420ms (Threshold: 200ms).")

# Demonstrating propagate = False
print("\n--- Disabling propagation on db_logger ---")
db_logger.propagate = False
db_logger.error("This log will NOT appear because propagation is disabled and db_logger has no handlers of its own!")


Auth Logger Name: app.api.auth
Parent Logger Name: app
Effective Level of Auth Logger: DEBUG

[PARENT HANDLER] app.api.auth | INFO | User 'alice' authenticated successfully.
[PARENT HANDLER] app.database | WARNING | Query execution took 420ms (Threshold: 200ms).

--- Disabling propagation on db_logger ---
This log will NOT appear because propagation is disabled and db_logger has no handlers of its own!


## 📌 Module 3: Formatting & Lazy String Evaluation

### Standard Formatter Specifiers
`logging.Formatter` allows formatting using standard attributes:
- `%(asctime)s`: Human-readable time when the `LogRecord` was created.
- `%(name)s`: Name of the logger.
- `%(levelname)s`: Text logging level (`INFO`, `ERROR`, etc.).
- `%(filename)s`: Source file basename where the log call was issued.
- `%(lineno)d`: Source line number where the log call was issued.
- `%(funcName)s`: Function name containing the log call.
- `%(process)d`: Process ID.
- `%(threadName)s`: Thread name.

### Crucial Performance Rule: Eager `f-strings` vs. Lazy String Interpolation
❌ **Bad Practice**: `logger.debug(f"User payload: {expensive_json_serialize(data)}")`
- The `f-string` is evaluated **eagerly**, even if the logger's level is `INFO` and `DEBUG` logs are ignored!

✅ **Best Practice**: `logger.debug("User payload: %s", data)`
- String formatting is **deferred** (lazy) and executed ONLY if the log record passes level filters.


In [4]:
import time

reset_logging()

logger = logging.getLogger("perf_demo")
logger.setLevel(logging.INFO) # DEBUG logs will be ignored!

def expensive_computation():
    print("  [!] WARNING: Eager function executed!")
    time.sleep(0.05)
    return {"user_id": 99, "data": "x" * 1000}

print("1. Eager Evaluation with f-string (Level = INFO, Log = DEBUG):")
start = time.perf_counter()
# f-string forces computation even though DEBUG is disabled!
logger.debug(f"Computed payload: {expensive_computation()}")
eager_duration = time.perf_counter() - start
print(f"   Elapsed Time: {eager_duration*1000:.2f} ms\n")

print("2. Lazy Interpolation with %s (Level = INFO, Log = DEBUG):")
start = time.perf_counter()
# Passing args lazily prevents execution of expensive string conversion when log level is skipped
logger.debug("Computed payload: %s", "some_data")
lazy_duration = time.perf_counter() - start
print(f"   Elapsed Time: {lazy_duration*1000:.2f} ms")

print("\n=> Lazy formatting avoided unnecessary overhead!")


1. Eager Evaluation with f-string (Level = INFO, Log = DEBUG):
  [!] WARNING: Eager function executed!
   Elapsed Time: 53.36 ms

2. Lazy Interpolation with %s (Level = INFO, Log = DEBUG):
   Elapsed Time: 0.00 ms

=> Lazy formatting avoided unnecessary overhead!


## 📌 Module 4: Handlers & Multi-Destination Logging (Console + Log Rotation)

In real applications, you often want logs routed simultaneously to multiple handlers:
1. **Console (`StreamHandler`)**: Shows `INFO` and above for real-time observation.
2. **Rotating File (`RotatingFileHandler`)**: Captures `DEBUG` and above to disk, rotating files when they reach a max size (e.g. 5 MB) with up to $N$ backups.
3. **Time-Based Rotation (`TimedRotatingFileHandler`)**: Rotates logs at set intervals (midnight, hourly, daily).


In [5]:
from logging.handlers import RotatingFileHandler, TimedRotatingFileHandler
import os

reset_logging()

# Setup log directory
log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)
app_log_file = os.path.join(log_dir, "app.log")

# Create logger
app_logger = logging.getLogger("enterprise_app")
app_logger.setLevel(logging.DEBUG)

# Handler 1: Console (INFO level + short format)
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)
console_formatter = logging.Formatter('📱 [CONSOLE] %(levelname)s: %(message)s')
console_handler.setFormatter(console_formatter)

# Handler 2: Rotating File Handler (DEBUG level + detailed format)
# Rotates after 1 KB for demo purposes, keeping 3 backup files (app.log.1, app.log.2, etc.)
file_handler = RotatingFileHandler(app_log_file, maxBytes=1024, backupCount=3, encoding='utf-8')
file_handler.setLevel(logging.DEBUG)
file_formatter = logging.Formatter('%(asctime)s | %(name)s | %(levelname)-8s | [%(filename)s:%(lineno)d] | %(message)s')
file_handler.setFormatter(file_formatter)

# Attach both handlers
app_logger.addHandler(console_handler)
app_logger.addHandler(file_handler)

# Emit messages
app_logger.debug("DEBUG: Initializing memory pool allocation.")
app_logger.info("INFO: Application server started on port 8080.")
app_logger.warning("WARNING: API rate limit warning for client 192.168.1.50.")
app_logger.error("ERROR: Failed to write to cache cluster.")

# Generate enough logs to trigger rotation
for i in range(30):
    app_logger.debug(f"Synthetic transaction trace #{i:03d} - Payload byte buffer check ok.")

print("\n--- Disk File Verification ---")
created_files = os.listdir(log_dir)
for filename in sorted(created_files):
    filepath = os.path.join(log_dir, filename)
    print(f"File: {filename:15s} | Size: {os.path.getsize(filepath)} bytes")


📱 [CONSOLE] INFO: INFO: Application server started on port 8080.
📱 [CONSOLE] WARNING: WARNING: API rate limit warning for client 192.168.1.50.
📱 [CONSOLE] ERROR: ERROR: Failed to write to cache cluster.

--- Disk File Verification ---
File: app.log         | Size: 675 bytes
File: app.log.1       | Size: 945 bytes
File: app.log.2       | Size: 945 bytes
File: app.log.3       | Size: 945 bytes


## 📌 Module 5: Custom Formatters & Structured Logging (JSON & Colored Logs)

Modern cloud environments (AWS CloudWatch, Datadog, ELK, GCP Cloud Logging) parse logs best when formatted as **JSON lines** (`jsonlines`). 

For local terminal development, **Colored Output** improves developer readability.

Let's build both:
1. `ColoredFormatter` for ANSI terminal colorizing.
2. `JSONFormatter` for structured, machine-readable logs.
3. Adding custom dynamic context with `extra` dict and `LoggerAdapter`.


In [6]:
import json
from datetime import datetime, timezone

reset_logging()

# 1. Custom ANSI Colored Formatter
class ColoredFormatter(logging.Formatter):
    """Custom formatter providing ANSI color-coded log levels for terminal output."""
    COLORS = {
        'DEBUG': '\033[36m',    # Cyan
        'INFO': '\033[32m',     # Green
        'WARNING': '\033[33m',  # Yellow
        'ERROR': '\033[31m',    # Red
        'CRITICAL': '\033[41m\033[37m' # Red background + White text
    }
    RESET = '\033[0m'

    def format(self, record):
        color = self.COLORS.get(record.levelname, self.RESET)
        record.levelname_color = f"{color}{record.levelname:8s}{self.RESET}"
        formatter = logging.Formatter('%(asctime)s | %(levelname_color)s | %(name)s - %(message)s', datefmt='%H:%M:%S')
        return formatter.format(record)

# 2. Custom JSON Formatter for Production Observability
class JSONFormatter(logging.Formatter):
    """Custom formatter outputting standardized JSON record structures."""
    def format(self, record):
        log_obj = {
            "timestamp": datetime.fromtimestamp(record.created, tz=timezone.utc).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
            "source": {
                "file": record.filename,
                "line": record.lineno,
                "function": record.funcName
            }
        }
        # Include extra context if available
        if hasattr(record, "request_id"):
            log_obj["request_id"] = record.request_id
        if hasattr(record, "user_id"):
            log_obj["user_id"] = record.user_id
        if record.exc_info:
            log_obj["exception"] = self.formatException(record.exc_info)
        return json.dumps(log_obj)

# Demonstrate Colored Logging
colored_handler = logging.StreamHandler(sys.stdout)
colored_handler.setFormatter(ColoredFormatter())
dev_logger = logging.getLogger("dev_app")
dev_logger.addHandler(colored_handler)
dev_logger.setLevel(logging.DEBUG)

print("--- 1. Colored Terminal Logs ---")
dev_logger.info("Developer workstation initialized.")
dev_logger.warning("DeprecationWarning: feature 'x' will be removed in v2.0.")
dev_logger.error("Connection failed to localhost:5432.")

# Demonstrate JSON Structured Logging
reset_logging()
json_handler = logging.StreamHandler(sys.stdout)
json_handler.setFormatter(JSONFormatter())
prod_logger = logging.getLogger("prod_service")
prod_logger.addHandler(json_handler)
prod_logger.setLevel(logging.INFO)

print("\n--- 2. Production Structured JSON Logs ---")
# Passing dynamic context using the `extra` argument
prod_logger.info("Order processed successfully.", extra={"request_id": "req-987654", "user_id": "usr-42"})


--- 1. Colored Terminal Logs ---
16:34:42 | INFO     | dev_app - Developer workstation initialized.
16:34:42 | WARNING  | dev_app - DeprecationWarning: feature 'x' will be removed in v2.0.
16:34:42 | ERROR    | dev_app - Connection failed to localhost:5432.

--- 2. Production Structured JSON Logs ---
{"timestamp": "2026-09-09T21:34:42.409520+00:00", "level": "INFO", "logger": "prod_service", "message": "Order processed successfully.", "source": {"file": "<string>", "line": 70, "function": "<module>"}, "request_id": "req-987654", "user_id": "usr-42"}


## 📌 Module 6: Enterprise Config with `dictConfig` & Robust Exception Tracking

### Declarative Configuration with `logging.config.dictConfig`
In production frameworks (FastAPI, Django, Flask, Celery), logging is configured declaratively using Python dictionaries (or loaded from YAML/JSON files).

### Capturing Exception Context
Never log errors as plain strings when an exception occurs!
- `logger.error("An error occurred")` -> Missing traceback!
- `logger.exception("An error occurred")` -> Automatically captures the active exception traceback!
- `logger.error("An error occurred", exc_info=True)` -> Equivalent to `logger.exception()`.


In [7]:
import logging.config

reset_logging()

# Enterprise Logging Dictionary Configuration
LOGGING_CONFIG = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "standard": {
            "format": "%(asctime)s [%(levelname)s] %(name)s: %(message)s"
        },
        "detailed": {
            "format": "%(asctime)s | %(levelname)-7s | %(name)s | %(filename)s:%(lineno)d | %(message)s"
        }
    },
    "handlers": {
        "console": {
            "class": "logging.StreamHandler",
            "level": "INFO",
            "formatter": "standard",
            "stream": "ext://sys.stdout"
        },
        "file_errors": {
            "class": "logging.FileHandler",
            "level": "ERROR",
            "formatter": "detailed",
            "filename": "logs/errors.log",
            "mode": "a"
        }
    },
    "loggers": {
        "api_server": {
            "level": "DEBUG",
            "handlers": ["console", "file_errors"],
            "propagate": False
        }
    }
}

# Apply dictionary config
logging.config.dictConfig(LOGGING_CONFIG)

logger = logging.getLogger("api_server")
logger.info("API Server initialized with dictConfig.")

print("\n--- Exception Handling Demonstration ---")
def calculate_tax(price, rate):
    try:
        return price / rate
    except ZeroDivisionError:
        # logger.exception automatically appends exc_info traceback!
        logger.exception("Tax calculation failed due to division by zero! Price: %s, Rate: %s", price, rate)

calculate_tax(100.0, 0.0)

# Verify error log file contents
if os.path.exists("logs/errors.log"):
    print("\n--- Extracted Content from logs/errors.log ---")
    with open("logs/errors.log", "r") as f:
        print(f.read().strip())


2026-09-09 16:34:42,443 [INFO] api_server: API Server initialized with dictConfig.

--- Exception Handling Demonstration ---
2026-09-09 16:34:42,443 [ERROR] api_server: Tax calculation failed due to division by zero! Price: 100.0, Rate: 0.0
Traceback (most recent call last):
  File "<string>", line 50, in calculate_tax
ZeroDivisionError: division by zero

--- Extracted Content from logs/errors.log ---
2026-09-09 16:34:42,443 | ERROR   | api_server | <string>:53 | Tax calculation failed due to division by zero! Price: 100.0, Rate: 0.0
Traceback (most recent call last):
  File "<string>", line 50, in calculate_tax
ZeroDivisionError: division by zero


## 📌 Module 7: Real-World ETL Pipeline Project with Audit Verification

Let's synthesize everything into a complete, realistic production simulation:
An **ETL (Extract, Transform, Load) Pipeline** that processes data records, handles corrupted rows gracefully, logs audit events in JSON format, rotates log files, and tracks performance execution metrics.


In [8]:
import json
import time
import os
import logging
from logging.handlers import RotatingFileHandler
from datetime import datetime, timezone

reset_logging()

class ETLPipeline:
    """Production ETL Pipeline with integrated structured logging and rotation."""
    
    def __init__(self, pipeline_name: str, log_dir: str = "etl_logs"):
        self.pipeline_name = pipeline_name
        self.log_dir = log_dir
        os.makedirs(self.log_dir, exist_ok=True)
        
        self.logger = self._setup_logging()
        
    def _setup_logging(self) -> logging.Logger:
        logger = logging.getLogger(self.pipeline_name)
        logger.setLevel(logging.DEBUG)
        
        # Prevent adding handlers multiple times
        if logger.handlers:
            return logger
            
        # Console Handler (JSON format)
        class PipelineJSONFormatter(logging.Formatter):
            def format(self, record):
                data = {
                    "timestamp": datetime.fromtimestamp(record.created, tz=timezone.utc).isoformat(),
                    "pipeline": record.name,
                    "level": record.levelname,
                    "event": record.getMessage(),
                }
                if hasattr(record, "record_id"):
                    data["record_id"] = record.record_id
                if hasattr(record, "execution_time_ms"):
                    data["execution_time_ms"] = record.execution_time_ms
                if record.exc_info:
                    data["exception"] = self.formatException(record.exc_info)
                return json.dumps(data)
                
        console_h = logging.StreamHandler(sys.stdout)
        console_h.setLevel(logging.INFO)
        console_h.setFormatter(PipelineJSONFormatter())
        
        # Rotating File Handler for audit trail
        audit_file = os.path.join(self.log_dir, "pipeline_audit.log")
        file_h = RotatingFileHandler(audit_file, maxBytes=5000, backupCount=2, encoding='utf-8')
        file_h.setLevel(logging.DEBUG)
        file_h.setFormatter(PipelineJSONFormatter())
        
        logger.addHandler(console_h)
        logger.addHandler(file_h)
        return logger

    def process_records(self, dataset: list):
        self.logger.info(f"Starting ETL job for dataset containing {len(dataset)} items.")
        successful = 0
        failed = 0
        start_job = time.perf_counter()
        
        for item in dataset:
            rec_id = item.get("id", "UNKNOWN")
            start_item = time.perf_counter()
            try:
                # Validation & Transformation step
                if "val" not in item:
                    raise KeyError("Missing required field 'val'")
                
                transformed_val = float(item["val"]) * 1.15 # 15% tax adjustment
                elapsed_ms = round((time.perf_counter() - start_item) * 1000, 3)
                
                self.logger.debug(
                    "Record transformed successfully.",
                    extra={"record_id": rec_id, "execution_time_ms": elapsed_ms}
                )
                successful += 1
                
            except Exception as e:
                failed += 1
                self.logger.error(
                    f"Record transformation failed: {str(e)}",
                    extra={"record_id": rec_id},
                    exc_info=True
                )
                
        total_time_ms = round((time.perf_counter() - start_job) * 1000, 2)
        self.logger.info(
            f"ETL Job Complete. Success: {successful}, Failed: {failed}",
            extra={"execution_time_ms": total_time_ms}
        )
        return {"success": successful, "failed": failed, "total_time_ms": total_time_ms}

# Execute the ETL Pipeline with mock data
mock_data = [
    {"id": "REC-001", "val": "100.50"},
    {"id": "REC-002", "val": "250.00"},
    {"id": "REC-003"}, # Malformed - missing 'val' field!
    {"id": "REC-004", "val": "invalid_number"}, # Malformed - non-numeric!
    {"id": "REC-005", "val": "75.25"}
]

pipeline = ETLPipeline(pipeline_name="SalesDataETL")
metrics = pipeline.process_records(mock_data)

print("\n--- Pipeline Audit Log Sample (from etl_logs/pipeline_audit.log) ---")
audit_path = "etl_logs/pipeline_audit.log"
if os.path.exists(audit_path):
    with open(audit_path, "r") as f:
        lines = f.readlines()
        for l in lines[:3]: # Print first 3 log lines
            print(l.strip())


{"timestamp": "2026-09-09T21:34:42.444741+00:00", "pipeline": "SalesDataETL", "level": "INFO", "event": "Starting ETL job for dataset containing 5 items."}
{"timestamp": "2026-09-09T21:34:42.444955+00:00", "pipeline": "SalesDataETL", "level": "ERROR", "event": "Record transformation failed: \"Missing required field 'val'\"", "record_id": "REC-003", "exception": "Traceback (most recent call last):\n  File \"<string>\", line 71, in process_records\nKeyError: \"Missing required field 'val'\""}
{"timestamp": "2026-09-09T21:34:42.445188+00:00", "pipeline": "SalesDataETL", "level": "ERROR", "event": "Record transformation failed: could not convert string to float: 'invalid_number'", "record_id": "REC-004", "exception": "Traceback (most recent call last):\n  File \"<string>\", line 73, in process_records\nValueError: could not convert string to float: 'invalid_number'"}
{"timestamp": "2026-09-09T21:34:42.445386+00:00", "pipeline": "SalesDataETL", "level": "INFO", "event": "ETL Job Complete. S

## 📌 Module 8: Cheatsheet, Best Practices & Production Anti-Patterns

### 💡 Quick Reference Cheatsheet

| Component | Standard Tool | When to Use |
| :--- | :--- | :--- |
| **Console Output** | `logging.StreamHandler(sys.stdout)` | Real-time debugging in local dev or Docker stdout |
| **File Rotation (Size)** | `RotatingFileHandler(filename, maxBytes, backupCount)` | Production servers to avoid filling up disk space |
| **File Rotation (Time)** | `TimedRotatingFileHandler(filename, when='midnight')` | Daily compliance & log archiving requirements |
| **Declarative Setup** | `logging.config.dictConfig(config_dict)` | Standard practice for FastAPI, Django, Flask apps |
| **Exception Logging** | `logger.exception("Message")` | Inside `except` blocks to capture stack traces |
| **Context Injection** | `logger.info("Msg", extra={"request_id": id})` | Correlation IDs in distributed microservices |

---

### ✅ Production Best Practices:
1. **Always use `__name__` as Logger Name**: `logger = logging.getLogger(__name__)`. This ensures logger names mirror your package/module structure (`myapp.services.auth`).
2. **Use Lazy String Formatting**: Write `logger.info("User %s created", user)` instead of `f-strings` or `%` formatting inside logger calls to avoid unnecessary string evaluation overhead.
3. **Log Exceptions with Context**: Always use `logger.exception()` or `exc_info=True` inside `except` blocks.
4. **Log Structured Data in Production**: Format logs as JSON lines (`JSONFormatter`) for seamless ingestion into ELK, Datadog, Splunk, or AWS CloudWatch.
5. **Set Handlers & Loggers Independently**: Keep logger levels broad (e.g. `DEBUG`) and restrict handler levels (e.g. `StreamHandler` = `INFO`, `FileHandler` = `DEBUG`).

### ❌ Anti-Patterns to Avoid:
- ❌ **Never use `print()` in production libraries or apps.**
- ❌ **Never use `logger.info(f"...")` for heavy computation or debug logs.**
- ❌ **Never swallow exceptions silently without logging (`except Exception: pass`).**
- ❌ **Never re-add handlers without clearing old ones in interactive environments (leads to duplicate log lines).**
